In [1]:
from pathlib import Path
from datetime import datetime
import gc
import json
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

PROJECT = Path("/Users/yw/unsw/Kaggle")
TARGET = "Will_Buy_EV"

SEED = 42

# 先做 5-fold，没必要一上来 10-fold
OUTER_FOLDS = 5

# 每个 outer training fold 内部再做 3-fold，
# 防止 Window Target Encoding 泄漏
INNER_FOLDS = 3

THREADS = max(
    1,
    (os.cpu_count() or 8) - 1,
)

TRAIN_PATH = (
    PROJECT
    / "data/train.csv"
)

TEST_PATH = (
    PROJECT
    / "data/test.csv"
)

SAMPLE_PATH = (
    PROJECT
    / "data/sample_submission.csv"
)

train = pd.read_csv(
    TRAIN_PATH
)

test = pd.read_csv(
    TEST_PATH
)

sample = (
    pd.read_csv(
        SAMPLE_PATH
    )
    if SAMPLE_PATH.exists()
    else None
)

if TARGET not in train.columns:

    raise ValueError(
        f"Missing target {TARGET}. "
        f"Columns: {train.columns.tolist()}"
    )

y = (
    train[TARGET]
    .map(
        {
            "No": 0,
            "Yes": 1,
        }
    )
    .astype(np.int8)
)

OUT_DIR = (
    PROJECT
    / "outputs"
    / (
        "v10_window_xgb_"
        + datetime.now().strftime(
            "%Y%m%d_%H%M%S_%f"
        )
    )
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

# 比 V8 稍微轻一点。
# 重点是 feature view，不是继续疯狂堆树。
XGB_PARAMS = dict(

    n_estimators=7000,

    learning_rate=0.025,

    max_depth=5,

    min_child_weight=8,

    subsample=0.82,

    colsample_bytree=0.82,

    reg_lambda=6.0,

    reg_alpha=0.05,

    gamma=0.0,

    max_bin=512,

    objective=
        "binary:logistic",

    eval_metric="auc",

    tree_method="hist",

    early_stopping_rounds=250,

    n_jobs=THREADS,
)

print(
    "train:",
    train.shape,
)

print(
    "test:",
    test.shape,
)

print(
    "positive rate:",
    float(y.mean()),
)

print(
    "threads:",
    THREADS,
)

print(
    "output:",
    OUT_DIR,
)

train: (668665, 15)
test: (286571, 14)
positive rate: 0.17464500160768098
threads: 9
output: /Users/yw/unsw/Kaggle/outputs/v10_window_xgb_20261001_030543_973913


In [2]:
# ============================================================
# V10 feature view
# row-local + leakage-safe window target encoding
# ============================================================

BASE_FEATURES = [

    "Age",

    "Gender",

    "Annual_Income_USD",

    "City_Type",

    "Daily_Commute_km",

    "Number_of_Cars_Owned",

    "Current_Car_Type",

    "Charging_Stations_Near_Home",

    "Charging_Stations_Near_Work",

    "Home_Charging_Possible",

    "Environmental_Concern_Level",

    "Subsidy_Available",

    "Range_Anxiety_Level",
]


# window 是“附近多少个整数坐标”
#
# Income:
# ±50 / ±250 / ±1000 / ±5000 USD
#
# Commute:
# scale=10
# ±1 就是 ±0.1 km
# ±5 就是 ±0.5 km
# ±20 就是 ±2 km
#
WINDOW_SPECS = {

    "Annual_Income_USD": {

        "scale": 1,

        "windows": [
            0,
            50,
            250,
            1000,
            5000,
        ],
    },

    "Daily_Commute_km": {

        "scale": 10,

        "windows": [
            0,
            1,
            5,
            20,
            50,
        ],
    },

    "Age": {

        "scale": 1,

        "windows": [
            0,
            1,
            3,
            5,
        ],
    },

    "Number_of_Cars_Owned": {

        "scale": 1,

        "windows": [
            0,
            1,
        ],
    },

    "Charging_Stations_Near_Home": {

        "scale": 1,

        "windows": [
            0,
            1,
            2,
        ],
    },

    "Charging_Stations_Near_Work": {

        "scale": 1,

        "windows": [
            0,
            1,
            2,
        ],
    },

    "Environmental_Concern_Level": {

        "scale": 1,

        "windows": [
            0,
            1,
        ],
    },
}


def yes_no_binary(
    series,
):

    if pd.api.types.is_numeric_dtype(
        series
    ):

        return (
            pd.to_numeric(
                series,
                errors="coerce",
            )
            .fillna(0)
            .astype(np.float32)
        )

    return (

        series
        .astype(str)
        .str.strip()
        .str.lower()

        .map(
            {
                "yes": 1.0,
                "no": 0.0,

                "true": 1.0,
                "false": 0.0,

                "1": 1.0,
                "0": 0.0,
            }
        )

        .fillna(0.0)

        .astype(np.float32)
    )


# ============================================================
# Label-free row features
# ============================================================

def add_row_features(
    frame,
):

    missing = [

        col
        for col in BASE_FEATURES

        if col
        not in frame.columns
    ]

    if missing:

        raise ValueError(
            f"Missing features: {missing}"
        )

    x = (
        frame[
            BASE_FEATURES
        ]
        .copy()
    )

    numeric_cols = [

        "Age",

        "Annual_Income_USD",

        "Daily_Commute_km",

        "Number_of_Cars_Owned",

        "Charging_Stations_Near_Home",

        "Charging_Stations_Near_Work",

        "Environmental_Concern_Level",
    ]

    for col in numeric_cols:

        x[col] = pd.to_numeric(

            x[col],

            errors="coerce",
        )

    income = (
        x[
            "Annual_Income_USD"
        ]
        .fillna(0)
    )

    commute = (
        x[
            "Daily_Commute_km"
        ]
        .fillna(0)
    )

    cars = (
        x[
            "Number_of_Cars_Owned"
        ]
        .fillna(0)
    )

    home = (
        x[
            "Charging_Stations_Near_Home"
        ]
        .fillna(0)
    )

    work = (
        x[
            "Charging_Stations_Near_Work"
        ]
        .fillna(0)
    )

    concern = (
        x[
            "Environmental_Concern_Level"
        ]
        .fillna(0)
    )

    subsidy = yes_no_binary(
        frame[
            "Subsidy_Available"
        ]
    )

    # ----------------------------------
    # interactions
    # ----------------------------------

    x[
        "charging_total"
    ] = (
        home
        + work
    )

    x[
        "charging_gap"
    ] = (
        home
        - work
    )

    x[
        "income_per_car"
    ] = (
        income
        / (
            cars
            + 1.0
        )
    )

    x[
        "income_per_commute"
    ] = (
        income
        / (
            commute
            + 1.0
        )
    )

    x[
        "subsidy_bin"
    ] = subsidy

    x[
        "subsidy_x_income"
    ] = (
        subsidy
        * income
    )

    x[
        "subsidy_x_concern"
    ] = (
        subsidy
        * concern
    )

    x[
        "subsidy_x_home"
    ] = (
        subsidy
        * home
    )

    x[
        "subsidy_x_work"
    ] = (
        subsidy
        * work
    )

    # ----------------------------------
    # digit decomposition
    # ----------------------------------

    digit_specs = {

        "Age":
            (1, 2),

        "Annual_Income_USD":
            (1, 6),

        "Daily_Commute_km":
            (10, 3),

        "Number_of_Cars_Owned":
            (1, 1),

        "Charging_Stations_Near_Home":
            (1, 2),

        "Charging_Stations_Near_Work":
            (1, 2),

        "Environmental_Concern_Level":
            (1, 1),
    }

    for (
        col,
        (
            scale,
            positions,
        ),
    ) in digit_specs.items():

        values = np.rint(

            pd.to_numeric(
                frame[col],
                errors="coerce",
            )
            .fillna(0)
            .to_numpy()

            * scale

        ).astype(
            np.int64
        )

        for digit in range(
            positions
        ):

            x[
                f"{col}_digit_{digit}"
            ] = (

                np.abs(
                    values
                )

                // (
                    10 ** digit
                )

                % 10

            ).astype(
                np.int8
            )

    income_int = (
        np.rint(
            income
        )
        .astype(np.int64)
    )

    x[
        "income_last_2"
    ] = (
        income_int
        % 100
    )

    x[
        "income_last_3"
    ] = (
        income_int
        % 1000
    )

    x[
        "income_prefix_1000"
    ] = (
        income_int
        // 1000
    )

    # ----------------------------------
    # known structural thresholds
    # only as FEATURES
    # ----------------------------------

    x[
        "income_31_42k"
    ] = (

        (
            income
            >= 31004
        )

        &

        (
            income
            <= 41970
        )

    ).astype(
        np.int8
    )

    x[
        "income_ge_170537"
    ] = (
        income
        >= 170537
    ).astype(
        np.int8
    )

    x[
        "commute_ge_83"
    ] = (
        commute
        >= 83
    ).astype(
        np.int8
    )

    return x


# ============================================================
# Numeric value → integer coordinate
# ============================================================

def numeric_key(
    series,
    scale,
):

    values = (

        pd.to_numeric(
            series,
            errors="coerce",
        )

        .fillna(0)

        .to_numpy()
    )

    return np.rint(

        values
        * scale

    ).astype(
        np.int64
    )


# ============================================================
# Build local target-stat window
# ============================================================

def make_window_mapping(
    keys,
    labels,
    windows,
    smoothing,
):

    keys = np.asarray(
        keys,
        dtype=np.int64,
    )

    labels = np.asarray(
        labels,
        dtype=np.float64,
    )

    prior = float(
        labels.mean()
    )

    stats = (

        pd.DataFrame(
            {
                "key":
                    keys,

                "y":
                    labels,
            }
        )

        .groupby(
            "key",
            sort=True,
        )["y"]

        .agg(
            [
                "sum",
                "count",
            ]
        )
    )

    min_key = int(
        stats.index.min()
    )

    max_key = int(
        stats.index.max()
    )

    full_index = np.arange(
        min_key,
        max_key + 1,
        dtype=np.int64,
    )

    sums = (

        stats["sum"]

        .reindex(
            full_index,
            fill_value=0.0,
        )

        .to_numpy(
            dtype=np.float64
        )
    )

    counts = (

        stats["count"]

        .reindex(
            full_index,
            fill_value=0.0,
        )

        .to_numpy(
            dtype=np.float64
        )
    )

    mappings = {}

    for window in windows:

        if window == 0:

            local_sum = sums

            local_count = counts

        else:

            width = (
                2
                * int(window)
                + 1
            )

            local_sum = (

                pd.Series(
                    sums
                )

                .rolling(
                    width,
                    center=True,
                    min_periods=1,
                )

                .sum()

                .to_numpy()
            )

            local_count = (

                pd.Series(
                    counts
                )

                .rolling(
                    width,
                    center=True,
                    min_periods=1,
                )

                .sum()

                .to_numpy()
            )

        rate = (

            local_sum
            + smoothing
            * prior

        ) / (

            local_count
            + smoothing
        )

        mappings[
            int(window)
        ] = {

            "min_key":
                min_key,

            "max_key":
                max_key,

            "rate":
                rate.astype(
                    np.float32
                ),

            "count":
                local_count.astype(
                    np.float32
                ),

            "prior":
                prior,
        }

    return mappings


# ============================================================
# Apply a mapping
# ============================================================

def apply_window_mapping(
    keys,
    mapping,
):

    keys = np.asarray(
        keys,
        dtype=np.int64,
    )

    min_key = (
        mapping[
            "min_key"
        ]
    )

    max_key = (
        mapping[
            "max_key"
        ]
    )

    rate = np.full(

        len(keys),

        mapping[
            "prior"
        ],

        dtype=np.float32,
    )

    count = np.zeros(

        len(keys),

        dtype=np.float32,
    )

    inside = (

        (keys >= min_key)

        &

        (keys <= max_key)
    )

    offsets = (

        keys[
            inside
        ]

        - min_key
    )

    rate[
        inside
    ] = (

        mapping[
            "rate"
        ][
            offsets
        ]
    )

    count[
        inside
    ] = (

        mapping[
            "count"
        ][
            offsets
        ]
    )

    return (
        rate,
        count,
    )


# ============================================================
# Inner-CV cross-fitted window features
# ============================================================

def add_window_features_crossfit(

    train_raw,

    train_y,

    valid_raw,

    test_raw,

    seed,
):

    tr = (
        train_raw
        .reset_index(
            drop=True
        )
    )

    va = (
        valid_raw
        .reset_index(
            drop=True
        )
    )

    te = (
        test_raw
        .reset_index(
            drop=True
        )
    )

    yy = np.asarray(
        train_y,
        dtype=np.int8,
    )

    out_train = pd.DataFrame(
        index=np.arange(
            len(tr)
        )
    )

    out_valid = pd.DataFrame(
        index=np.arange(
            len(va)
        )
    )

    out_test = pd.DataFrame(
        index=np.arange(
            len(te)
        )
    )

    inner_folds = list(

        StratifiedKFold(

            n_splits=
                INNER_FOLDS,

            shuffle=True,

            random_state=
                seed,

        ).split(
            tr,
            yy,
        )
    )

    for (
        col,
        spec,
    ) in WINDOW_SPECS.items():

        scale = (
            spec[
                "scale"
            ]
        )

        windows = (
            spec[
                "windows"
            ]
        )

        train_keys = (
            numeric_key(
                tr[col],
                scale,
            )
        )

        valid_keys = (
            numeric_key(
                va[col],
                scale,
            )
        )

        test_keys = (
            numeric_key(
                te[col],
                scale,
            )
        )

        for window in windows:

            smoothing = (
                20.0
                if window == 0
                else 50.0
            )

            oof_rate = np.full(

                len(tr),

                np.nan,

                dtype=np.float32,
            )

            oof_count = np.full(

                len(tr),

                np.nan,

                dtype=np.float32,
            )

            for (
                fit_indices,
                hold_indices,
            ) in inner_folds:

                mapping = (

                    make_window_mapping(

                        train_keys[
                            fit_indices
                        ],

                        yy[
                            fit_indices
                        ],

                        windows=[
                            window
                        ],

                        smoothing=
                            smoothing,

                    )[
                        window
                    ]
                )

                (
                    rate,
                    count,
                ) = apply_window_mapping(

                    train_keys[
                        hold_indices
                    ],

                    mapping,
                )

                oof_rate[
                    hold_indices
                ] = rate

                oof_count[
                    hold_indices
                ] = count

            if not np.isfinite(
                oof_rate
            ).all():

                raise ValueError(
                    f"{col} window {window}: "
                    f"bad OOF encoding"
                )

            full_mapping = (

                make_window_mapping(

                    train_keys,

                    yy,

                    windows=[
                        window
                    ],

                    smoothing=
                        smoothing,

                )[
                    window
                ]
            )

            (
                valid_rate,
                valid_count,
            ) = apply_window_mapping(

                valid_keys,

                full_mapping,
            )

            (
                test_rate,
                test_count,
            ) = apply_window_mapping(

                test_keys,

                full_mapping,
            )

            tag = (
                f"{col}"
                f"_win{window}"
            )

            out_train[
                f"{tag}_te"
            ] = oof_rate

            out_valid[
                f"{tag}_te"
            ] = valid_rate

            out_test[
                f"{tag}_te"
            ] = test_rate

            # 模型同时知道这个窗口有多少训练样本支持
            out_train[
                f"{tag}_logcount"
            ] = np.log1p(
                oof_count
            ).astype(
                np.float32
            )

            out_valid[
                f"{tag}_logcount"
            ] = np.log1p(
                valid_count
            ).astype(
                np.float32
            )

            out_test[
                f"{tag}_logcount"
            ] = np.log1p(
                test_count
            ).astype(
                np.float32
            )

    return (
        out_train,
        out_valid,
        out_test,
    )


# ============================================================
# Fold-local category encoding
# ============================================================

def encode_fold_categories(

    train_x,

    valid_x,

    test_x,
):

    a = (
        train_x
        .copy()
        .reset_index(
            drop=True
        )
    )

    b = (
        valid_x
        .copy()
        .reset_index(
            drop=True
        )
    )

    c = (
        test_x
        .copy()
        .reset_index(
            drop=True
        )
    )

    categorical_cols = [

        col

        for col
        in a.columns

        if (

            pd.api.types
            .is_object_dtype(
                a[col]
            )

            or

            pd.api.types
            .is_string_dtype(
                a[col]
            )

            or

            isinstance(
                a[col].dtype,
                pd.CategoricalDtype,
            )
        )
    ]

    for col in categorical_cols:

        train_values = (
            a[col]
            .astype(str)
        )

        categories = pd.Index(
            pd.unique(
                train_values
            )
        )

        mapping = {

            value:
                index

            for index, value
            in enumerate(
                categories
            )
        }

        for frame in (
            a,
            b,
            c,
        ):

            frame[col] = (

                frame[col]
                .astype(str)

                .map(
                    mapping
                )

                .fillna(
                    -1
                )

                .astype(
                    np.int32
                )
            )

    for frame in (
        a,
        b,
        c,
    ):

        for col in frame.columns:

            if (
                pd.api.types
                .is_bool_dtype(
                    frame[col]
                )
            ):

                frame[col] = (
                    frame[col]
                    .astype(
                        np.int8
                    )
                )

        bad = (

            frame
            .select_dtypes(
                include=[
                    "object",
                    "string",
                    "category",
                ]
            )
            .columns
            .tolist()
        )

        if bad:

            raise ValueError(
                f"Non-numeric columns remain: {bad}"
            )

    return (
        a,
        b,
        c,
    )

In [3]:
# ============================================================
# Train V10 Window-Encoding XGBoost
# ============================================================

base_train = add_row_features(
    train
)

base_test = add_row_features(
    test
)

folds = list(

    StratifiedKFold(

        n_splits=
            OUTER_FOLDS,

        shuffle=True,

        random_state=
            SEED,

    ).split(
        train,
        y,
    )
)

oof = np.full(

    len(train),

    np.nan,

    dtype=np.float64,
)

coverage = np.zeros(

    len(train),

    dtype=np.int8,
)

test_sum = np.zeros(

    len(test),

    dtype=np.float64,
)

fold_rows = []


for (
    fold,
    (
        train_indices,
        valid_indices,
    ),
) in enumerate(
    folds,
    start=1,
):

    print()

    print(
        "=" * 74
    )

    print(
        f"V10 Window-XGB | "
        f"Fold {fold}/{OUTER_FOLDS}"
    )

    print(
        "=" * 74
    )

    raw_train = (

        train
        .iloc[
            train_indices
        ]

        .reset_index(
            drop=True
        )
    )

    raw_valid = (

        train
        .iloc[
            valid_indices
        ]

        .reset_index(
            drop=True
        )
    )

    raw_test = (

        test
        .reset_index(
            drop=True
        )
    )

    x_train = (

        base_train
        .iloc[
            train_indices
        ]

        .reset_index(
            drop=True
        )
    )

    x_valid = (

        base_train
        .iloc[
            valid_indices
        ]

        .reset_index(
            drop=True
        )
    )

    x_test = (

        base_test
        .reset_index(
            drop=True
        )
    )

    # --------------------------------
    # Leakage-safe Window Encoding
    # --------------------------------

    (
        window_train,
        window_valid,
        window_test,
    ) = add_window_features_crossfit(

        raw_train,

        y.iloc[
            train_indices
        ].reset_index(
            drop=True
        ),

        raw_valid,

        raw_test,

        seed=
            1000
            + fold,
    )

    x_train = pd.concat(
        [
            x_train,
            window_train,
        ],
        axis=1,
    )

    x_valid = pd.concat(
        [
            x_valid,
            window_valid,
        ],
        axis=1,
    )

    x_test = pd.concat(
        [
            x_test,
            window_test,
        ],
        axis=1,
    )

    # --------------------------------
    # Category encoding
    # --------------------------------

    (
        x_train,
        x_valid,
        x_test,
    ) = encode_fold_categories(

        x_train,

        x_valid,

        x_test,
    )

    if (
        x_train.columns.tolist()
        !=
        x_valid.columns.tolist()
    ):

        raise ValueError(
            "train/valid columns differ"
        )

    if (
        x_train.columns.tolist()
        !=
        x_test.columns.tolist()
    ):

        raise ValueError(
            "train/test columns differ"
        )

    # --------------------------------
    # XGBoost
    # --------------------------------

    model_params = dict(
        XGB_PARAMS
    )

    model_params[
        "random_state"
    ] = (
        SEED
        + fold
    )

    model = XGBClassifier(
        **model_params
    )

    model.fit(

        x_train,

        y.iloc[
            train_indices
        ],

        eval_set=[
            (
                x_valid,

                y.iloc[
                    valid_indices
                ],
            )
        ],

        verbose=250,
    )

    valid_prediction = (

        model
        .predict_proba(
            x_valid
        )[:, 1]
    )

    test_prediction = (

        model
        .predict_proba(
            x_test
        )[:, 1]
    )

    oof[
        valid_indices
    ] = valid_prediction

    coverage[
        valid_indices
    ] += 1

    test_sum += (
        test_prediction
    )

    fold_auc = float(

        roc_auc_score(

            y.iloc[
                valid_indices
            ],

            valid_prediction,
        )
    )

    best_iteration = getattr(

        model,

        "best_iteration",

        None,
    )

    fold_rows.append(
        {
            "fold":
                fold,

            "auc":
                fold_auc,

            "best_iteration":
                (
                    None

                    if best_iteration
                    is None

                    else int(
                        best_iteration
                    )
                ),

            "n_features":
                x_train.shape[1],
        }
    )

    pd.DataFrame(
        fold_rows
    ).to_csv(

        OUT_DIR
        / "fold_scores_window_xgb.csv",

        index=False,
    )

    np.savez_compressed(

        OUT_DIR
        / (
            f"window_xgb_"
            f"fold{fold}_"
            f"predictions.npz"
        ),

        valid_indices=
            valid_indices,

        valid_ids=
            train["id"]
            .iloc[
                valid_indices
            ]
            .to_numpy(),

        valid_prediction=
            valid_prediction,

        test_ids=
            test["id"]
            .to_numpy(),

        test_prediction=
            test_prediction,
    )

    model.save_model(

        str(

            OUT_DIR
            / (
                f"window_xgb_"
                f"fold{fold}.json"
            )
        )
    )

    print(

        f"Fold {fold} "
        f"AUC={fold_auc:.9f} | "

        f"best_iteration="
        f"{best_iteration} | "

        f"features="
        f"{x_train.shape[1]}"
    )

    del (
        raw_train,
        raw_valid,
        raw_test,

        x_train,
        x_valid,
        x_test,

        window_train,
        window_valid,
        window_test,

        model,

        valid_prediction,
        test_prediction,
    )

    gc.collect()


if not np.all(
    coverage == 1
):

    raise ValueError(
        "OOF coverage incomplete"
    )


if not np.isfinite(
    oof
).all():

    raise ValueError(
        "OOF contains NaN/inf"
    )


test_prediction = (
    test_sum
    / OUTER_FOLDS
)


overall_auc = float(

    roc_auc_score(
        y,
        oof,
    )
)


print()

print(
    "=" * 74
)

print(

    pd.DataFrame(
        fold_rows
    )

    .to_string(

        index=False,

        float_format=
            lambda value:
                f"{value:.9f}",
    )
)

print()

print(

    f"V10 Window-XGB "
    f"overall OOF AUC "
    f"= {overall_auc:.9f}"
)

print(
    "=" * 74
)


V10 Window-XGB | Fold 1/5
[0]	validation_0-auc:0.93712
[250]	validation_0-auc:0.94423
[500]	validation_0-auc:0.94462
[750]	validation_0-auc:0.94465
[1000]	validation_0-auc:0.94462
[1110]	validation_0-auc:0.94462
Fold 1 AUC=0.944662182 | best_iteration=860 | features=93

V10 Window-XGB | Fold 2/5
[0]	validation_0-auc:0.93728
[250]	validation_0-auc:0.94492
[500]	validation_0-auc:0.94528
[750]	validation_0-auc:0.94530
[955]	validation_0-auc:0.94519
Fold 2 AUC=0.945316161 | best_iteration=705 | features=93

V10 Window-XGB | Fold 3/5
[0]	validation_0-auc:0.93982
[250]	validation_0-auc:0.94602
[500]	validation_0-auc:0.94649
[750]	validation_0-auc:0.94659
[1000]	validation_0-auc:0.94659
[1220]	validation_0-auc:0.94654
Fold 3 AUC=0.946601245 | best_iteration=970 | features=93

V10 Window-XGB | Fold 4/5
[0]	validation_0-auc:0.93915
[250]	validation_0-auc:0.94551
[500]	validation_0-auc:0.94574
[750]	validation_0-auc:0.94574
[902]	validation_0-auc:0.94566
Fold 4 AUC=0.945769764 | best_iteration=

In [5]:
# ============================================================
# Save V10
# ============================================================

oof_path = (
    OUT_DIR
    / "oof_window_xgb.csv"
)

submission_path = (
    OUT_DIR
    / "submission_window_xgb.csv"
)


pd.DataFrame(
    {
        "id":
            train["id"]
            .to_numpy(),

        "target":
            y.to_numpy(),

        "prediction":
            oof,
    }
).to_csv(
    oof_path,
    index=False,
)


submission = pd.DataFrame(
    {
        "id":
            test["id"]
            .to_numpy(),

        TARGET:
            test_prediction,
    }
)


if sample is not None:

    submission = (

        sample[
            ["id"]
        ]

        .merge(

            submission,

            on="id",

            how="left",

            validate=
                "one_to_one",

            sort=False,
        )
    )


submission.to_csv(
    submission_path,
    index=False,
)


# ============================================================
# Read previous engines
# ============================================================

def read_aligned_oof(
    path,
):

    frame = pd.read_csv(
        path
    )

    return (

        frame
        .set_index(
            "id"
        )

        .loc[
            train["id"],
            "prediction",
        ]

        .to_numpy(
            dtype=np.float64
        )
    )


OLD_XGB_DIR = (

    PROJECT
    / "outputs"
    / "v5_xgb10_20260930_205349_440980"
)


old_xgb_oof = read_aligned_oof(

    OLD_XGB_DIR
    / "oof_xgboost.csv"
)


# ============================================================
# Find newest V8 strong model
# ============================================================

v8_directories = sorted(

    (
        PROJECT
        / "outputs"
    ).glob(
        "v8_strong_xgb_*"
    ),

    key=
        lambda path:
            path.stat().st_mtime,

    reverse=True,
)


strong_rich_oof = None
strong_rich_dir = None


for directory in v8_directories:

    path = (

        directory
        / "oof_strong_rich_te.csv"
    )

    if path.is_file():

        strong_rich_oof = (
            read_aligned_oof(
                path
            )
        )

        strong_rich_dir = (
            directory
        )

        break


def spearman(
    a,
    b,
):

    return float(

        pd.Series(a)
        .corr(

            pd.Series(b),

            method=
                "spearman",
        )
    )


correlation_rows = [

    {
        "comparison":
            "V10 vs old_XGB",

        "spearman":
            spearman(
                oof,
                old_xgb_oof,
            ),
    }
]


if strong_rich_oof is not None:

    correlation_rows.append(
        {
            "comparison":
                "V10 vs strong_rich_te",

            "spearman":
                spearman(
                    oof,
                    strong_rich_oof,
                ),
        }
    )


correlation_table = (
    pd.DataFrame(
        correlation_rows
    )
)


print(
    "V10 OOF:",
    overall_auc,
)

print()

print(

    correlation_table

    .to_string(

        index=False,

        float_format=
            lambda value:
                f"{value:.9f}",
    )
)


# ============================================================
# Quick V5 blend test
# ============================================================

CAT_DIR = (

    PROJECT
    / "outputs"
    / "v3_20260925_180457_278867"
)

LGB_DIR = (

    PROJECT
    / "outputs"
    / "v4_lightgbm_20260930_163526_079778"
)


cat_oof = read_aligned_oof(

    CAT_DIR
    / "oof_catboost.csv"
)

lgb_oof = read_aligned_oof(

    LGB_DIR
    / "oof_lightgbm.csv"
)


def rank01(
    values,
):

    return (

        pd.Series(
            values
        )

        .rank(
            method="average",
            pct=True,
        )

        .to_numpy(
            dtype=np.float64
        )
    )


current_oof = (

    0.5
    * cat_oof

    +

    0.5
    * lgb_oof
)


v5_oof = (

    0.5
    * rank01(
        old_xgb_oof
    )

    +

    0.5
    * rank01(
        current_oof
    )
)


v5_auc = float(

    roc_auc_score(
        y,
        v5_oof,
    )
)


v10_rank = rank01(
    oof
)

v5_rank = rank01(
    v5_oof
)


blend_rows = []


for v10_weight in [

    0.10,
    0.20,
    0.30,
    0.40,
    0.50,

]:

    prediction = (

        v10_weight
        * v10_rank

        +

        (
            1.0
            - v10_weight
        )

        * v5_rank
    )


    auc = float(

        roc_auc_score(
            y,
            prediction,
        )
    )


    blend_rows.append(
        {
            "candidate":

                f"V10_"
                f"{int(v10_weight*100)}"

                f"__V5_"
                f"{int((1-v10_weight)*100)}",

            "oof_auc":
                auc,

            "delta_vs_v5":
                auc
                - v5_auc,
        }
    )


blend_table = (

    pd.DataFrame(
        blend_rows
    )

    .sort_values(
        "oof_auc",
        ascending=False,
    )

    .reset_index(
        drop=True
    )
)


print()

print(
    "V5 OOF:",
    v5_auc,
)

print()

print(

    blend_table

    .to_string(

        index=False,

        float_format=
            lambda value:
                f"{value:.9f}",
    )
)


report = {

    "v10_oof_auc":
        overall_auc,

    "v5_oof_auc":
        v5_auc,

    "correlations":
        correlation_rows,

    "fold_scores":
        fold_rows,

    "strong_rich_dir":
        (
            None

            if strong_rich_dir
            is None

            else str(
                strong_rich_dir
            )
        ),
}


(
    OUT_DIR
    / "v10_report.json"
).write_text(

    json.dumps(
        report,
        indent=2,
    ),

    encoding="utf-8",
)


print()

print(
    "Saved:"
)

print(
    oof_path
)

print(
    submission_path
)

print(
    OUT_DIR
    / "v10_report.json"
)

V10 OOF: 0.9455160201935802

           comparison    spearman
       V10 vs old_XGB 0.994363836
V10 vs strong_rich_te 0.994620826

V5 OOF: 0.9460987453734608

    candidate     oof_auc  delta_vs_v5
V10_10__V5_90 0.946109184  0.000010439
V10_20__V5_80 0.946104515  0.000005770
V10_30__V5_70 0.946084656 -0.000014089
V10_40__V5_60 0.946049474 -0.000049271
V10_50__V5_50 0.945999126 -0.000099620

Saved:
/Users/yw/unsw/Kaggle/outputs/v10_window_xgb_20261001_030543_973913/oof_window_xgb.csv
/Users/yw/unsw/Kaggle/outputs/v10_window_xgb_20261001_030543_973913/submission_window_xgb.csv
/Users/yw/unsw/Kaggle/outputs/v10_window_xgb_20261001_030543_973913/v10_report.json
